# Artificial Intelligence — Lab 4
## Greedy Best-First Search and A*

**Course Learning Outcomes — CLO2 and CLO3**

- **CLO2:** Determine appropriate uninformed and heuristic search strategies.
- **CLO3:** Illustrate and analyze the performance of search methods.

**Environment:** Python 3 / Jupyter Notebook  
**Submission:** completed notebook containing predictions, traces, code, experiments, justifications, debugging answers, and reflection.

> **Assessment principle:** Working code is only one part of the evidence. Most marks come from your ability to **reason about heuristics, predict node ordering, justify $f(n)$ values, compare solution quality and search effort, and explain why A* can outperform or correct Greedy Best-First Search**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Heuristic reasoning | 15 min | Interpret $h(n)$ and compare evaluation functions |
| 2. Manual Greedy and A* traces | 25 min | Predict frontier ordering before coding |
| 3. Implement both algorithms | 35 min | Use priority queues with different priorities |
| 4. Compare search behavior | 20 min | Analyze cost, expansions, and heuristic influence |
| 5. Grid heuristic experiment | 15 min | Compare weak and stronger heuristics |
| 6. Debugging, variation & reflection | 10 min | Diagnose errors and defend conclusions |

> **Main idea:** Greedy Best-First Search uses only estimated remaining cost, while A* combines cost already paid with estimated cost to go.

## Learning Objectives

By the end of this lab, you should be able to:

1. explain the meaning of a heuristic function $h(n)$;
2. distinguish the evaluation functions of Greedy Best-First Search and A*;
3. manually trace both algorithms on a small weighted graph;
4. implement both algorithms using a priority queue;
5. compute and interpret

$$
f_{\text{Greedy}}(n)=h(n)
$$

and

$$
f_{A^*}(n)=g(n)+h(n);
$$

6. explain admissibility and consistency conceptually;
7. compare path cost, expanded nodes, and frontier size;
8. investigate how heuristic quality affects search effort;
9. diagnose common heuristic-search implementation errors;
10. justify when A* is preferable to Greedy Best-First Search.

In [ ]:
import heapq
from itertools import count
from typing import Dict, List, Tuple, Optional
from math import inf

print("Lab 4 environment ready.")

# Part I — Heuristics and Evaluation Functions

A heuristic estimates the remaining cost from a state to a goal:

$$
h(n)\approx h^*(n)
$$

where $h^*(n)$ is the true optimal remaining cost.

We will use this graph:

```text
S --2--> A --2--> G
 \
  1
   \
    B --1--> C --10--> G
```

Heuristic values:

| State | $h(n)$ |
|---|---:|
| S | 3 |
| A | 2 |
| B | 1 |
| C | 1 |
| G | 0 |

Important: the heuristic makes the branch through `B` look attractive even though its final path is expensive.

In [ ]:
GRAPH = {
    "S": [("A", 2), ("B", 1)],
    "A": [("G", 2)],
    "B": [("C", 1)],
    "C": [("G", 10)],
    "G": [],
}

H = {
    "S": 3,
    "A": 2,
    "B": 1,
    "C": 1,
    "G": 0,
}

START = "S"
GOAL = "G"

## Task 1.1 — Compute Candidate Path Costs

Compute:

| Path | Number of actions | Total cost |
|---|---:|---:|
| $S \rightarrow A \rightarrow G$ |  |  |
| $S \rightarrow B \rightarrow C \rightarrow G$ |  |  |

Then answer:

1. Which path is cheapest?
2. Which branch initially looks more promising according to $h$ after expanding `S`?
3. Why can a heuristic be useful without being perfect?

**Your answers:**

## Task 1.2 — Compare Greedy and A* Priorities

After expanding `S`, compute:

### Greedy Best-First Search

$$
f(n)=h(n)
$$

| Node | $h(n)$ | Greedy priority |
|---|---:|---:|
| A |  |  |
| B |  |  |

### A*

$$
f(n)=g(n)+h(n)
$$

| Node | $g(n)$ | $h(n)$ | $f(n)$ |
|---|---:|---:|---:|
| A |  |  |  |
| B |  |  |  |

Then answer:

1. Which node does Greedy prefer?
2. Which node does A* prefer initially?
3. Why can the two algorithms still behave differently later even if they initially select the same node?

**Your answers:**

# Part II — Manual Greedy Best-First Search Trace

Greedy Best-First Search chooses the frontier node with the smallest heuristic value:

$$
f(n)=h(n).
$$

It ignores the path cost already paid.

## Task 2.1 — Predict the Greedy Trace

Complete the trace before coding.

| Step | Expanded node | $h$ | Frontier ordered by $h$ |
|---:|---|---:|---|
| 0 | — | — | `[(S,3)]` |
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

Then answer:

1. What path do you predict Greedy will return?
2. What will its total path cost be?
3. Is this path optimal for this graph?
4. Why does Greedy make this choice?

**Your answers:**

# Part III — Manual A* Trace

A* chooses the frontier node with the smallest

$$
f(n)=g(n)+h(n).
$$

It balances:

- **cost already paid**: $g(n)$;
- **estimated remaining cost**: $h(n)$.

## Task 3.1 — Predict the A* Trace

Complete:

| Step | Expanded node | $g$ | $h$ | $f=g+h$ | Frontier after expansion |
|---:|---|---:|---:|---:|---|
| 0 | — | — | — | — | `[(S,3)]` |
| 1 |  |  |  |  |  |
| 2 |  |  |  |  |  |
| 3 |  |  |  |  |  |

Then answer:

1. What path do you predict A* will return?
2. What will its total cost be?
3. At what point does A* avoid being trapped by the misleading heuristic?

**Your answers:**

# Part IV — Implement Greedy Best-First Search

The function should return:

```python
(path, total_cost, stats)
```

where `stats` records expansion order and basic performance metrics.

In [ ]:
def reconstruct_path(parent: Dict[str, Optional[str]], goal: str) -> List[str]:
    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = parent[current]
    return list(reversed(path))


def path_cost(graph, path):
    if path is None:
        return inf

    total = 0
    for current, nxt in zip(path, path[1:]):
        for child, step_cost in graph[current]:
            if child == nxt:
                total += step_cost
                break
        else:
            raise ValueError(f"No edge {current} -> {nxt}")
    return total

## Task 4.1 — Complete Greedy Best-First Search

Use a priority queue ordered only by `heuristic[state]`.

### Required behavior

- priority = $h(n)$;
- maintain parent links;
- avoid repeatedly inserting already discovered states;
- stop when the goal is popped;
- record expanded nodes and maximum frontier size.

In [ ]:
def greedy_best_first_search(graph, heuristic, start, goal):
    tie = count()
    frontier = [(heuristic[start], next(tie), start)]
    discovered = {start}
    parent = {start: None}

    expanded_nodes = 0
    max_frontier_size = 1
    expansion_order = []

    while frontier:
        # TODO 1: pop the lowest-h node
        h_value, _, current = None, None, None

        # TODO 2: record expansion

        # TODO 3: if goal, reconstruct path and return

        # TODO 4:
        # for each child:
        #   if not discovered:
        #       mark discovered
        #       set parent
        #       push with priority heuristic[child]

        # TODO 5: update max_frontier_size
        pass

    return None, inf, {
        "expanded_nodes": expanded_nodes,
        "max_frontier_size": max_frontier_size,
        "expansion_order": expansion_order,
    }

### Task 4.2 — Predict Before Running

Write first:

- **Expected Greedy path:**  
- **Expected cost:**  
- **Expected first three expansions:**  

Then run the self-check.

In [ ]:
greedy_path, greedy_cost, greedy_stats = greedy_best_first_search(
    GRAPH, H, START, GOAL
)

print("Greedy path:", greedy_path)
print("Greedy cost:", greedy_cost)
print("Expansion order:", greedy_stats["expansion_order"])
print("Expanded nodes:", greedy_stats["expanded_nodes"])
print("Maximum frontier size:", greedy_stats["max_frontier_size"])

assert greedy_path == ["S", "B", "C", "G"]
assert greedy_cost == 12

print("Greedy tests passed.")

## Task 4.3 — Justify Greedy Behavior

Answer:

1. Why did Greedy choose `B` before `A`?
2. Why did it then choose `C`?
3. Why does Greedy not account for the large final edge cost into `G` until it is too late?
4. Does a small $h(n)$ necessarily mean a small total solution cost? Explain.

**Your answers:**

# Part V — Implement A*

A* uses

$$
f(n)=g(n)+h(n).
$$

The implementation needs both:

- a priority queue ordered by $f$;
- a table of the best known $g$ cost to each state.

## Task 5.1 — Complete A*

### Required behavior

- start with $g(S)=0$;
- priority = $g(n)+h(n)$;
- store best-known $g$ cost;
- update a parent when a cheaper path is found;
- ignore stale heap entries;
- stop when the goal is popped.

In [ ]:
def astar_search(graph, heuristic, start, goal):
    tie = count()

    frontier = [(heuristic[start], next(tie), 0, start)]
    best_g = {start: 0}
    parent = {start: None}

    expanded_nodes = 0
    max_frontier_size = 1
    expansion_order = []

    while frontier:
        # Heap entry: (f, tie_breaker, g, state)
        f_value, _, g_value, current = None, None, None, None

        # TODO 1: pop the lowest-f entry

        # TODO 2: skip stale entries:
        # if g_value != best_g[current]: continue

        # TODO 3: record (current, g, h, f)

        # TODO 4: if current == goal:
        # return path, g_value, stats

        # TODO 5:
        # for child, step_cost in graph[current]:
        #   tentative_g = g_value + step_cost
        #   if tentative_g is better than best_g.get(child, inf):
        #       update best_g
        #       update parent
        #       child_f = tentative_g + heuristic[child]
        #       push new heap entry

        # TODO 6: update maximum frontier size
        pass

    return None, inf, {
        "expanded_nodes": expanded_nodes,
        "max_frontier_size": max_frontier_size,
        "expansion_order": expansion_order,
    }

### Task 5.2 — Predict Before Running

Write:

- **Expected A* path:**  
- **Expected cost:**  
- **Expected first three expansions:**  

Then run the self-check.

In [ ]:
astar_path, astar_cost, astar_stats = astar_search(
    GRAPH, H, START, GOAL
)

print("A* path:", astar_path)
print("A* cost:", astar_cost)
print("Expansion order:", astar_stats["expansion_order"])
print("Expanded nodes:", astar_stats["expanded_nodes"])
print("Maximum frontier size:", astar_stats["max_frontier_size"])

assert astar_path == ["S", "A", "G"]
assert astar_cost == 4

print("A* tests passed.")

## Task 5.3 — Explain the A* Implementation

Answer in your own words.

1. What does `best_g[state]` represent?
2. Why is $f(n)$ not stored as the only measure of path quality?
3. Why must A* keep the actual $g(n)$ separately?
4. What is a stale A* heap entry?
5. Why does A* update a state when a cheaper $g$ path is found?

**Your answers:**

# Part VI — Compare Greedy and A*

Run both algorithms on the same graph and compare their behavior.

In [ ]:
greedy_path, greedy_cost, greedy_stats = greedy_best_first_search(
    GRAPH, H, START, GOAL
)

astar_path, astar_cost, astar_stats = astar_search(
    GRAPH, H, START, GOAL
)

print("GREEDY")
print("  path:", greedy_path)
print("  cost:", greedy_cost)
print("  expanded:", greedy_stats["expanded_nodes"])
print("  max frontier:", greedy_stats["max_frontier_size"])

print("\nA*")
print("  path:", astar_path)
print("  cost:", astar_cost)
print("  expanded:", astar_stats["expanded_nodes"])
print("  max frontier:", astar_stats["max_frontier_size"])

## Task 6.1 — Interpret the Comparison

Complete:

| Metric | Greedy | A* |
|---|---:|---:|
| Returned path |  |  |
| Path cost |  |  |
| Expanded nodes |  |  |
| Maximum frontier size |  |  |

Then explain:

1. Which algorithm returned the lower-cost solution?
2. Why did Greedy fail to return the optimal solution here?
3. Why did A* recover from the misleading branch?
4. If Greedy expanded fewer nodes in some problem, would that automatically make it the better algorithm? Why not?
5. Which metric represents **solution quality**, and which represent **search effort**?

**Your answers:**

# Part VII — Admissibility and Consistency

A heuristic is **admissible** if

$$
0 \le h(n)\le h^*(n)
$$

for every state.

A heuristic is **consistent** if

$$
h(n)\le c(n,a,n')+h(n')
$$

for every transition.

For this graph, compute the true optimal remaining costs manually if needed.

## Task 7.1 — Check the Heuristic

For each state, estimate or calculate $h^*(n)$.

| State | Given $h(n)$ | True optimal remaining cost $h^*(n)$ | Admissible here? |
|---|---:|---:|---|
| S | 3 |  |  |
| A | 2 |  |  |
| B | 1 |  |  |
| C | 1 |  |  |
| G | 0 |  |  |

Then answer:

1. Is the heuristic admissible for all states?
2. Does admissibility require the heuristic to be exact?
3. Can an admissible heuristic still be weak?

**Your answers:**

## Task 7.2 — Check One Consistency Inequality

For edge $S \rightarrow A$ with cost 2, test:

$$
h(S)\le 2+h(A).
$$

For edge $B \rightarrow C$ with cost 1, test:

$$
h(B)\le 1+h(C).
$$

Show the numerical inequalities and state whether they hold.

**Your answer:**

# Part VIII — Heuristics on a Grid

We now use a 4-directional grid.

For a goal at $(r_g,c_g)$, Manhattan distance is

$$
h_{\text{Manhattan}}(n)=|r-r_g|+|c-c_g|.
$$

We will compare:

- zero heuristic: $h(n)=0$;
- Manhattan heuristic.

With $h(n)=0$, A* behaves like Uniform-Cost Search.

In [ ]:
GRID = [
    "........",
    ".###....",
    "...#....",
    "...#.#..",
    "........",
    "........",
]

GRID_START = (5, 0)
GRID_GOAL = (0, 7)

MOVES = [
    ("Up", (-1, 0)),
    ("Right", (0, 1)),
    ("Down", (1, 0)),
    ("Left", (0, -1)),
]

def grid_neighbors(state):
    r, c = state
    result = []
    for _, (dr, dc) in MOVES:
        nr, nc = r + dr, c + dc
        if (
            0 <= nr < len(GRID)
            and 0 <= nc < len(GRID[0])
            and GRID[nr][nc] != "#"
        ):
            result.append(((nr, nc), 1))
    return result

def h_zero(state, goal):
    return 0

def h_manhattan(state, goal):
    r, c = state
    gr, gc = goal
    return abs(r - gr) + abs(c - gc)

## Task 8.1 — Complete Grid A*

Adapt A* to use `grid_neighbors`.

In [ ]:
def astar_grid(start, goal, heuristic_fn):
    tie = count()

    frontier = [(heuristic_fn(start, goal), next(tie), 0, start)]
    best_g = {start: 0}
    parent = {start: None}

    expanded_nodes = 0
    max_frontier_size = 1

    while frontier:
        # TODO: complete the A* loop
        pass

    return None, inf, {
        "expanded_nodes": expanded_nodes,
        "max_frontier_size": max_frontier_size,
    }


def reconstruct_grid_path(parent, goal):
    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = parent[current]
    return list(reversed(path))

## Task 8.2 — Predict the Heuristic Effect

Before execution:

1. Will zero-heuristic A* and Manhattan A* return paths of the same optimal cost?
2. Which do you expect to expand fewer states?
3. Why?

**Your prediction:**

In [ ]:
zero_path, zero_cost, zero_stats = astar_grid(
    GRID_START, GRID_GOAL, h_zero
)

man_path, man_cost, man_stats = astar_grid(
    GRID_START, GRID_GOAL, h_manhattan
)

print("A* with h=0")
print("  path cost:", zero_cost)
print("  expanded:", zero_stats["expanded_nodes"])
print("  max frontier:", zero_stats["max_frontier_size"])

print("\nA* with Manhattan heuristic")
print("  path cost:", man_cost)
print("  expanded:", man_stats["expanded_nodes"])
print("  max frontier:", man_stats["max_frontier_size"])

## Task 8.3 — Analyze Heuristic Quality

Complete:

| Measure | $h=0$ | Manhattan |
|---|---:|---:|
| Path cost |  |  |
| Expanded nodes |  |  |
| Maximum frontier size |  |  |

Then answer:

1. Did both heuristics preserve optimality in this experiment?
2. Which heuristic provided more guidance?
3. Why does $h=0$ reduce A* to UCS?
4. Why is a more informative admissible heuristic generally useful?
5. Does a stronger heuristic guarantee lower runtime on every machine and every problem? Explain carefully.

**Your answers:**

# Part IX — Debugging Heuristic Search

## Task 9.1 — Faulty Greedy Priority

A student writes:

```python
priority = g + h
```

inside Greedy Best-First Search.

1. What algorithm is this moving toward?
2. What should Greedy use instead?
3. Why is the distinction conceptually important?

**Your answer:**

## Task 9.2 — Faulty A*: Ignoring $g$

A student writes:

```python
heapq.heappush(frontier, (heuristic[child], child))
```

inside A*.

1. What information is missing?
2. What priority should be used?
3. Why could this make A* behave like Greedy?

**Your answer:**

## Task 9.3 — Faulty Heuristic

Suppose someone uses:

$$
h(n)=100
$$

for every non-goal state in a graph where actual remaining costs are much smaller.

1. Is this guaranteed admissible?
2. Why can overestimation threaten the usual A* optimality guarantee?
3. Would $h(n)=0$ be admissible if all step costs are nonnegative?

**Your answer:**

# Part X — Personalized Heuristic Variation

Use the last digit of your student ID.

- `0–3`: set $h(B)=3$
- `4–6`: set $h(A)=1$
- `7–9`: set $h(C)=8$

Create a copy of `H` and modify only your assigned heuristic value.

In [ ]:
LAST_DIGIT = None  # TODO: replace with an integer from 0 to 9

personal_h = dict(H)

# TODO:
# Apply your assigned heuristic modification.
# Keep H unchanged.

## Task 10.1 — Predict Before Running

Write:

- **Modified heuristic value:**  
- **Predicted Greedy path:**  
- **Predicted A* path:**  
- **Do you expect either expansion order to change? Why?**  

Then run both algorithms.

In [ ]:
if LAST_DIGIT is not None:
    p_greedy_path, p_greedy_cost, p_greedy_stats = greedy_best_first_search(
        GRAPH, personal_h, START, GOAL
    )

    p_astar_path, p_astar_cost, p_astar_stats = astar_search(
        GRAPH, personal_h, START, GOAL
    )

    print("Personal Greedy:")
    print(" path:", p_greedy_path)
    print(" cost:", p_greedy_cost)
    print(" order:", p_greedy_stats["expansion_order"])

    print("\nPersonal A*:")
    print(" path:", p_astar_path)
    print(" cost:", p_astar_cost)
    print(" order:", p_astar_stats["expansion_order"])

## Task 10.2 — Explain the Personalized Result

1. Was your predicted Greedy path correct?
2. Was your predicted A* path correct?
3. Did the changed heuristic affect the state space, the edge costs, or only search guidance?
4. Did the modification preserve admissibility? Justify numerically if possible.
5. Why can changing only $h(n)$ alter expansion order without changing the underlying problem?

**Your answers:**

# Part XI — Individual Understanding Check

Your instructor may ask one short question about your notebook.

Possible prompts:

- Show me where Greedy uses only $h(n)$.
- Show me where A* calculates $g(n)+h(n)$.
- What does `best_g` represent?
- Why is $h=0$ equivalent to UCS inside A*?
- Why did Greedy return a more expensive solution?
- What does admissibility mean in your own words?
- How does Manhattan distance guide search?
- If I change $h(B)$, what part of the problem formulation changes?

> You are expected to explain the **AI concept represented by the code**, not memorize Python syntax.

# Reflection

Answer concisely but precisely.

### R1 — Greedy vs. A*
Why can Greedy be fast but risky?

**Answer:**

### R2 — A* Balance
Why does A* combine past cost and estimated future cost?

**Answer:**

### R3 — Admissibility
Why is admissibility connected to optimality?

**Answer:**

### R4 — Heuristic Strength
Why can two admissible heuristics produce very different numbers of expanded nodes?

**Answer:**

### R5 — Algorithm Selection
Give one scenario where Greedy may be acceptable and one where A* is preferable.

**Answer:**

# Submission Checklist

Before submitting, verify that your notebook contains:

- [ ] candidate-path cost calculations;
- [ ] Greedy priority calculations;
- [ ] A* $g$, $h$, and $f$ calculations;
- [ ] manual Greedy trace;
- [ ] manual A* trace;
- [ ] working Greedy implementation;
- [ ] working A* implementation;
- [ ] Greedy vs. A* comparison and interpretation;
- [ ] admissibility/consistency reasoning;
- [ ] grid A* comparison using $h=0$ and Manhattan distance;
- [ ] debugging answers;
- [ ] personalized heuristic variation;
- [ ] prediction made before the personalized run;
- [ ] reflection answers;
- [ ] visible outputs from important code cells.

Suggested filename:

```text
Lab04_StudentID.ipynb
```

# Assessment Guide — 10 Marks

| Component | Marks | Evidence expected |
|---|---:|---|
| **Correct implementation** | **2.0** | Greedy and A* work correctly |
| **Algorithmic justification** | **3.0** | Explains $h$, $g$, $f$, admissibility, best-known costs, and heuristic guidance |
| **Experimental analysis** | **2.0** | Interprets Greedy/A* and grid-heuristic results |
| **Trace / prediction / debugging** | **1.0** | Manual traces, predictions, and diagnosis of faulty heuristic-search logic |
| **Individual understanding check** | **1.0** | Short explanation of selected part of the student's own work |
| **Code quality & completeness** | **1.0** | Readable code, complete responses, required outputs |
| **Total** | **10.0** |  |

> **Key rule:** Correct code without an adequate explanation earns only a limited portion of the marks.

## Key Takeaways

- Greedy Best-First Search uses

$$
f(n)=h(n).
$$

- A* uses

$$
f(n)=g(n)+h(n).
$$

- Greedy can be strongly guided but may return an expensive solution.
- A* balances path cost already paid with estimated remaining cost.
- An admissible heuristic never overestimates the true remaining optimal cost.
- A stronger admissible heuristic can reduce unnecessary exploration.
- $h(n)=0$ makes A* behave like Uniform-Cost Search.
- Search evaluation should distinguish **solution quality** from **search effort**.

The next lab will move from systematic state-space search to **local search and hill climbing**.